# L01 · Introduction and environment self-check

This course moves from Genesis simulation to robot control, demonstrations, datasets, ACT/SmolVLA training, and closed-loop evaluation. L01 is a short entry check: select the project `.venv` kernel and use **Run All** once.

CPU is a valid path for the minimum L01–L06 exercises. When the reviewed ROCm stack and an AMD GPU are visible, this notebook uses them automatically.

## 1. Confirm the active environment

The summary should point to the project environment and show Python 3.12, Genesis 1.3.3, and an importable PyTorch. LeRobot is reported as a preview for later training lessons; it does not block the CPU fundamentals path.

In [ ]:
import json
import sys
from importlib.metadata import version

from robo_genesis.course_manifest import load_course_manifest
from robo_genesis.course_utils import environment_report
from robo_genesis.paths import OUTPUTS_DIR

lesson = load_course_manifest().lesson("L01")
assert lesson.duration_minutes == 30
assert lesson.status.value == "cpu-verified"

report = environment_report()
report["python_executable"] = sys.executable
report["course_package"] = version("robo-genesis-101")
print(f"{'python_executable':>24}: {sys.executable}")
print(f"{'course_package':>24}: {report['course_package']}")

if sys.version_info[:2] != (3, 12):
    raise RuntimeError("Use the project Python 3.12 environment, then restart the kernel.")
if report["genesis_world"] != "1.3.3":
    raise RuntimeError("Expected genesis-world==1.3.3; restore the locked environment.")
if str(report["torch"]).startswith("unavailable"):
    raise RuntimeError("PyTorch is not importable in this kernel; restore the project environment.")

try:
    import lerobot
    report["lerobot"] = version("lerobot")
    print(f"{'lerobot':>24}: {report['lerobot']} (needed later for training)")
except Exception as exc:
    report["lerobot"] = f"not ready ({type(exc).__name__})"
    print(f"{'lerobot':>24}: {report['lerobot']} — optional until the data/training lessons")

## 2. Select a backend and run one tensor operation

On ROCm, PyTorch deliberately uses names such as `torch.cuda` and `cuda:0` for an AMD GPU. The HIP value and device name identify the actual platform.

In [ ]:
import numpy as np
import torch
import genesis as gs

from robo_genesis.course_utils import select_backend, to_numpy

rocm_ready = bool(torch.version.hip) and bool(torch.cuda.is_available()) and hasattr(gs, "amdgpu")
backend = select_backend(prefer_rocm=True)
backend_name = "amdgpu" if rocm_ready else "cpu"
tensor_device = "cuda" if rocm_ready else "cpu"
tensor_result = (torch.arange(4, device=tensor_device, dtype=torch.float32).square() + 1)
tensor_values = tensor_result.detach().cpu().tolist()
assert tensor_values == [1.0, 2.0, 5.0, 10.0]
print(f"tensor device: {tensor_result.device}; values: {tensor_values}")

gs.init(backend=backend, seed=0, precision="32", logging_level="warning")

## 3. Run the smallest useful Genesis smoke

The sphere should fall under gravity. The next lesson explains these APIs; here they only confirm that Genesis can build and step real simulation state.

In [ ]:
scene = gs.Scene(
    show_viewer=False,
    sim_options=gs.options.SimOptions(dt=0.01, substeps=2),
)
scene.add_entity(gs.morphs.Plane(), name="ground")
sphere = scene.add_entity(
    gs.morphs.Sphere(radius=0.08, pos=(0.0, 0.0, 0.5)),
    name="falling_sphere",
)
scene.build()
position_before = to_numpy(sphere.get_pos()).reshape(-1)
for _ in range(20):
    scene.step()
position_after = to_numpy(sphere.get_pos()).reshape(-1)

assert position_before.shape == (3,) and position_after.shape == (3,)
assert np.isfinite(position_before).all() and np.isfinite(position_after).all()
assert float(position_after[2]) < float(position_before[2])
print(f"sphere z: {position_before[2]:.6f} m -> {position_after[2]:.6f} m")

## 4. Read the result and continue

A CPU result is sufficient for this lesson. If you need help, share the small JSON file together with the error message; you do not need to open or edit the report itself.

In [ ]:
warnings = []
if backend_name == "cpu":
    warnings.append("AMD ROCm was not selected; CPU is valid for the L01-L06 minimum path.")
if report["lerobot"] != "0.6.0":
    warnings.append("LeRobot 0.6.0 is not ready yet; install the training extra before L11.")

result = {
    "ok": True,
    "python": report["python"],
    "python_executable": report["python_executable"],
    "genesis_world": report["genesis_world"],
    "torch": str(report["torch"]),
    "torch_hip": report["torch_hip"],
    "lerobot": report["lerobot"],
    "backend": backend_name,
    "tensor_values": tensor_values,
    "sphere_z_before_m": float(position_before[2]),
    "sphere_z_after_m": float(position_after[2]),
    "warnings": warnings,
}
report_path = OUTPUTS_DIR / "l01" / "env_report.json"
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text(json.dumps(result, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

print("=" * 56)
print("ENVIRONMENT CHECK: PASSED")
print(f"Genesis backend: {backend_name}")
for warning in warnings:
    print(f"NOTE: {warning}")
print(f"Report: {report_path}")
print("Next: continue to L02.")